# Étape 3 : Évaluation finale et calibration

## Test set officiel + courbes + calibration (ECE)

**Métriques** : F1-Macro, AUPRC, MCC, coût IDA (FP×10 + FN×500)

---


## 0. Configuration


In [ ]:
import os
import sys
import time
import warnings
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.loader import load_aps_data, get_X_y, compute_total_cost, COST_FP, COST_FN
from src.utils.reproducibility import set_all_seeds, log_environment_info, setup_logging, get_cv_splitter
from src.evaluation.metrics import compute_all_metrics, print_metrics_report, find_optimal_threshold
from src.evaluation.plots import setup_plot_style, save_figure, COLORS

warnings.filterwarnings('ignore')
setup_logging(level=logging.INFO)
setup_plot_style()
%matplotlib inline

SEED = 42
set_all_seeds(SEED)
N_FOLDS = 5
cv = get_cv_splitter(n_splits=N_FOLDS, seed=SEED)

FIGURES_DIR = os.path.join(PROJECT_ROOT, 'reports', 'figures')
TABLES_DIR = os.path.join(PROJECT_ROOT, 'reports', 'tables')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')
for d in [FIGURES_DIR, TABLES_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
log_environment_info()
pd.show_versions()

import joblib
import xgboost as xgb
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
    matthews_corrcoef, f1_score, confusion_matrix,
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from src.evaluation.calibration import compute_ece, plot_reliability_diagram, calibrate_model


## 1. Chargement données et modèles


In [ ]:
train_df, test_df = load_aps_data(project_root=PROJECT_ROOT)
X_train, y_train = get_X_y(train_df)
X_test, y_test = get_X_y(test_df)

logreg = joblib.load(os.path.join(MODELS_DIR, 'logreg_elasticnet.pkl'))
rf_model = joblib.load(os.path.join(MODELS_DIR, 'random_forest.pkl'))
xgb_pack = joblib.load(os.path.join(MODELS_DIR, 'xgboost_best.pkl'))

def predict_xgb(pack, X):
    X_imp = pack['imputer'].transform(X)
    return pack['booster'].predict(xgb.DMatrix(X_imp))

models = {
    'Elastic Net': logreg,
    'Random Forest': rf_model,
    f"XGBoost {xgb_pack['strategy']}": xgb_pack,
}

def get_proba(name, model, X):
    if name.startswith('XGBoost'):
        return predict_xgb(model, X)
    return model.predict_proba(X)[:, 1]

print('Modèles chargés:', list(models.keys()))


## 2. Évaluation sur le test set


In [ ]:
test_results = []
predictions = {}

for name, model in models.items():
    proba = get_proba(name, model, X_test)
    thr, _, _, _ = find_optimal_threshold(y_train.values, get_proba(name, model, X_train), metric='mcc')
    pred = (proba >= thr).astype(int)
    m = compute_all_metrics(y_test.values, pred, proba)
    predictions[name] = {'proba': proba, 'pred': pred, 'threshold': thr}
    test_results.append({
        'Modèle': name, 'F1-Macro': m['F1-Macro'], 'AUPRC': m['AUPRC'], 'MCC': m['MCC'],
        'Precision': m['Precision'], 'Recall': m['Recall'], 'Coût IDA': m['Cost'], 'Seuil': thr,
    })
    print_metrics_report(y_test.values, pred, proba, model_name=f'{name} (test)')

test_df_metrics = pd.DataFrame(test_results)
display(test_df_metrics.round(4))
best_model_name = test_df_metrics.loc[test_df_metrics['MCC'].idxmax(), 'Modèle']
print(f'\nMeilleur modèle (MCC test) : {best_model_name}')


## 3. Courbes de performance (300 DPI)


In [ ]:
# 3.1 Precision-Recall
fig, ax = plt.subplots(figsize=(10, 7))
baseline = y_test.mean()
for name in models:
    p, r, _ = precision_recall_curve(y_test, predictions[name]['proba'])
    ap = average_precision_score(y_test, predictions[name]['proba'])
    ax.plot(r, p, lw=2, label=f'{name} (AUPRC={ap:.4f})')
ax.axhline(baseline, color='gray', ls='--', label=f'Aléatoire ({baseline:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Courbes Precision-Recall — Test set')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
save_figure(fig, 'pr_curves', save_dir=FIGURES_DIR)
plt.show()

# 3.2 ROC (référence)
fig, ax = plt.subplots(figsize=(10, 7))
for name in models:
    fpr, tpr, _ = roc_curve(y_test, predictions[name]['proba'])
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc(fpr, tpr):.4f})')
ax.plot([0, 1], [0, 1], 'k--')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('Courbes ROC — référence uniquement')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
save_figure(fig, 'roc_curves', save_dir=FIGURES_DIR)
plt.show()


In [ ]:
# 3.3 MCC vs seuil (meilleur modèle) — pas de 0.05
best_proba = predictions[best_model_name]['proba']
thresholds = np.arange(0.1, 0.91, 0.05)
mcc_scores, cost_scores = [], []
for t in thresholds:
    pred_t = (best_proba >= t).astype(int)
    mcc_scores.append(matthews_corrcoef(y_test, pred_t))
    cm = confusion_matrix(y_test, pred_t)
    tn, fp, fn, tp = cm.ravel()
    cost_scores.append(COST_FP * fp + COST_FN * fn)

best_mcc_idx = np.argmax(mcc_scores)
thr_mcc = thresholds[best_mcc_idx]
best_cost_idx = np.argmin(cost_scores)
thr_cost = thresholds[best_cost_idx]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresholds, mcc_scores, 'o-', color=COLORS['primary'])
ax.axvline(thr_mcc, color=COLORS['danger'], ls='--', label=f'opt MCC={thr_mcc:.2f}')
ax.set_xlabel('Seuil'); ax.set_ylabel('MCC'); ax.set_title(f'MCC vs seuil — {best_model_name}')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
save_figure(fig, 'mcc_threshold', save_dir=FIGURES_DIR)
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresholds, cost_scores, 'o-', color=COLORS['secondary'])
ax.axvline(thr_cost, color=COLORS['danger'], ls='--', label=f'opt coût={thr_cost:.2f}')
ax.set_xlabel('Seuil'); ax.set_ylabel('Coût IDA (€)'); ax.set_title('Coût métier vs seuil')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
save_figure(fig, 'cost_threshold', save_dir=FIGURES_DIR)
plt.show()
print(f'Seuil optimal MCC={thr_mcc:.2f} | Seuil optimal coût={thr_cost:.2f}')


In [ ]:
# 3.5 Barplot final
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ['F1-Macro', 'AUPRC', 'MCC']
for ax, met in zip(axes, metrics):
    ax.bar(test_df_metrics['Modèle'], test_df_metrics[met], color=COLORS['primary'], edgecolor='white')
    ax.set_title(met); ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
save_figure(fig, 'final_metrics_barplot', save_dir=FIGURES_DIR)
plt.show()


## 4. Calibration des probabilités


In [ ]:
# ECE avant calibration
ece_before = {}
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, model) in zip(axes, models.items()):
    proba = get_proba(name, model, X_test)
    prob_true, prob_pred = calibration_curve(y_test, proba, n_bins=10, strategy='uniform')
    ece = compute_ece(y_test.values, proba, n_bins=10)
    ece_before[name] = ece
    ax.plot([0, 1], [0, 1], 'k--')
    ax.plot(prob_pred, prob_true, 's-', label=f'ECE={ece:.4f}')
    ax.set_title(name); ax.set_xlabel('Proba prédite'); ax.set_ylabel('Fraction positifs')
    ax.legend()
plt.suptitle('Reliability diagrams — avant calibration', fontweight='bold')
plt.tight_layout()
save_figure(fig, 'calibration_before', save_dir=FIGURES_DIR)
plt.show()


In [ ]:
# Calibration — isotonic pour arbres/XGB, sigmoid (Platt) pour linéaire
proba_train_best = get_proba(best_model_name, models[best_model_name], X_train)
proba_test_best = predictions[best_model_name]['proba']
ece_before_val = ece_before[best_model_name]

if ece_before_val > 0.05:
    if best_model_name.startswith('XGBoost'):
        from sklearn.isotonic import IsotonicRegression
        print(f'Calibration isotonic (XGBoost) — ECE avant={ece_before_val:.4f}')
        iso_reg = IsotonicRegression(out_of_bounds='clip')
        iso_reg.fit(proba_train_best, y_train.values)
        proba_cal = iso_reg.transform(proba_test_best)
        calibrated = iso_reg
    else:
        cal_method = 'isotonic' if 'Forest' in best_model_name else 'sigmoid'
        print(f'Calibration CalibratedClassifierCV — méthode {cal_method}')
        calibrated = CalibratedClassifierCV(models[best_model_name], method=cal_method, cv=N_FOLDS)
        calibrated.fit(X_train, y_train)
        proba_cal = calibrated.predict_proba(X_test)[:, 1]
else:
    print(f'ECE={ece_before_val:.4f} <= 0.05 — calibration optionnelle, on garde les probas brutes')
    proba_cal = proba_test_best
    calibrated = None

ece_after = compute_ece(y_test.values, proba_cal, n_bins=10)
if calibrated is not None and best_model_name.startswith('XGBoost'):
    proba_cal_train = iso_reg.transform(proba_train_best)
elif calibrated is not None:
    proba_cal_train = calibrated.predict_proba(X_train)[:, 1]
else:
    proba_cal_train = proba_train_best
thr_cal, _, _, _ = find_optimal_threshold(y_train.values, proba_cal_train, metric='mcc')
pred_cal = (proba_cal >= thr_cal).astype(int)
m_cal = compute_all_metrics(y_test.values, pred_cal, proba_cal)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (proba, title) in zip(axes, [
    (predictions[best_model_name]['proba'], 'Avant'),
    (proba_cal, 'Après calibration'),
]):
    prob_true, prob_pred = calibration_curve(y_test, proba, n_bins=10)
    ece = compute_ece(y_test.values, proba, 10)
    ax.plot([0, 1], [0, 1], 'k--')
    ax.plot(prob_pred, prob_true, 's-', label=f'ECE={ece:.4f}')
    ax.set_title(f'{best_model_name} — {title}')
    ax.legend()
plt.tight_layout()
save_figure(fig, 'calibration_comparison', save_dir=FIGURES_DIR)
plt.show()

print(f'ECE avant : {ece_before[best_model_name]:.4f} | après : {ece_after:.4f}')
print(f'MCC avant : {test_df_metrics.loc[test_df_metrics["Modèle"]==best_model_name,"MCC"].values[0]:.4f} | après : {m_cal["MCC"]:.4f}')
if calibrated is not None:
    joblib.dump(calibrated, os.path.join(MODELS_DIR, 'best_model_calibrated.pkl'))


## 5. Synthèse rapport


In [ ]:
final_table = test_df_metrics.copy()
final_table['ECE'] = final_table['Modèle'].map(ece_before)
final_table['ECE après cal.'] = np.nan
final_table.loc[final_table['Modèle'] == best_model_name, 'ECE après cal.'] = ece_after
final_table['Seuil optimal MCC'] = final_table['Seuil']
final_table['Seuil coût minimal'] = thr_cost

display(final_table.round(4))
final_table.to_csv(os.path.join(TABLES_DIR, 'final_comparison.csv'), index=False)

print('''
**Recommandation production** :
- Modèle : ''' + best_model_name + '''
- Seuil MCC : ''' + f'{thr_mcc:.2f}' + ''' | Seuil coût : ''' + f'{thr_cost:.2f}' + '''
- Privilégier le seuil coût si l'objectif est le challenge IDA 2016.
- Appliquer la calibration si ECE > 0.05.
''')
